# 1 · Baseline (simple) PINN

Trains the baseline physics-informed neural network with **fixed hyperparameters** (no Bayesian optimisation) on the
training partition of a fixed three-way split (train / validation / test), then evaluates it on the validation and test
partitions.

**Before running:** place the input files in `../data/` as described in [`data/README.md`](../data/README.md).
To try the pipeline without the restricted field data, generate a synthetic dataset first:
`python examples/make_synthetic_data.py --out data` (from the repository root).

In [ ]:
import os, sys

# Make the repository root importable and point the module at the input data.
REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, REPO_ROOT)
DATA_DIR = os.path.join(REPO_ROOT, "data")      # see data/README.md for the required files
os.environ["DDE_BACKEND"] = "pytorch"           # must be set before deepxde is imported

import pinn_unified as pu

## Configuration

`SPLIT_TYPE` is one of `'random'`, `'temporal'` (contiguous mid-record gap) or `'temporal_lowdata'` (30 % training).

In [ ]:
SPLIT_TYPE = 'temporal'
USE_FILM = False
SPLIT_SEED = 42
SAVE_DIR = 'results_simple'

## 1. Load and scale the data

In [ ]:
stage_s, gwt_s, well_s, stats_df = pu.load_and_preprocess_data(data_dir=DATA_DIR)

## 2. Fixed three-way partition

In [ ]:
print(f"[INFO] Creating fixed 3-way partition: {SPLIT_TYPE}")
if SPLIT_TYPE == "random":
    GWT_TRAIN, GWT_VAL, GWT_TEST = pu.random_train_val_test_split(gwt_s, 0.7, 0.15, SPLIT_SEED)
elif SPLIT_TYPE == "temporal":
    GWT_TRAIN, GWT_VAL, GWT_TEST = pu.temporal_train_val_test_split(gwt_s, 0.7, 0.15, SPLIT_SEED)
elif SPLIT_TYPE == "temporal_lowdata":
    GWT_TRAIN, GWT_VAL, GWT_TEST = pu.temporal_train_val_test_split(gwt_s, 0.3, 0.15, SPLIT_SEED)
else:
    raise ValueError(SPLIT_TYPE)

## 3. Train the simple PINN (training partition only)

In [ ]:
print("[INFO] Starting simple PINN training...")
results = pu.run_pinn_iterations(
    stage_s=stage_s,
    gwt_s=GWT_TRAIN,
    well_s=well_s,
    stats_df=stats_df,
    et_rch_csv_path=pu.data_path('ET_RCH_VARUNA_ML.csv'),
    ztop_extn_csv_path=pu.data_path('Z_top_extn_depth.csv'),
    river_csv_path=pu.data_path('varuna_points_latlong.csv'),
    augment_river_dirichlet=True,
    include_forcings=False,
    use_film=USE_FILM,
    save_dir=SAVE_DIR,
)
print("Training complete!")

## 4. Evaluate on validation and test partitions

The baseline has no tuned hyperparameters, so evaluating it on the test partition does not leak test information into model selection.

In [ ]:
import torch
net, colmap, scales = results["final_net"], results["colmap"], results["scales"]
device = next(net.parameters()).device
for name, part in [("train", GWT_TRAIN), ("validation", GWT_VAL), ("test", GWT_TEST)]:
    pred, obs = pu._pred_obs_from_df_stream(net, part, colmap, scales, component=0,
                                            batch_size=4096, device=device, use_amp=False, pinned=False)
    print(f"{name:>10}: {pu.compute_basic_metrics(pred, obs)}")